# 🎯 수학 시험지 문제 영역 감지 — YOLOv8 학습

**목표**: 수학 시험지 PDF 페이지에서 문제/그래프/표 영역을 자동 감지

**클래스**:
- `0: problem` — 문제 전체 영역 (텍스트 + 수식 + 선택지)
- `1: graph` — 그래프/좌표평면/함수 그림
- `2: table` — 표/도표

**데이터 소스**: 자산화 시 수동 크롭된 bbox 데이터 (detection_annotations 테이블)

In [ ]:
# Cell 1: 환경 설정
!pip install -q ultralytics requests

import os
import json
import random
import requests
from pathlib import Path
from ultralytics import YOLO

print('✅ 환경 설정 완료')

# ⬇️ 여기에 실제 값 입력
EXPORT_API_URL = 'https://YOUR_DOMAIN/api/workflow/yolo-export?limit=5000'  # 수정 필요
# 또는 로컬에서 ngrok 사용 시:
# EXPORT_API_URL = 'http://localhost:3000/api/workflow/yolo-export?limit=5000'

In [ ]:
# Cell 2: 학습 데이터 다운로드

print('📥 Export API에서 매니페스트 다운로드 중...')
resp = requests.get(EXPORT_API_URL)
manifest = resp.json()

print(f"📊 전체 이미지: {manifest['totalImages']}")
print(f"📊 전체 어노테이션: {manifest['totalAnnotations']}")
print(f"📊 클래스 분포: {manifest.get('classDistribution', {})}")

if manifest['totalImages'] == 0:
    print('❌ 학습 데이터가 없습니다. 먼저 시험지를 자산화하세요!')
else:
    # YOLO 디렉토리 구조 생성
    dataset_dir = Path('dataset')
    for split in ['train', 'val']:
        (dataset_dir / split / 'images').mkdir(parents=True, exist_ok=True)
        (dataset_dir / split / 'labels').mkdir(parents=True, exist_ok=True)

    # 이미지 다운로드 + 라벨 저장
    items = manifest['manifest']
    random.shuffle(items)
    split_idx = int(len(items) * 0.85)  # 85% train, 15% val

    success = 0
    for i, item in enumerate(items):
        split = 'train' if i < split_idx else 'val'
        img_name = f'page_{i:04d}.png'
        label_name = f'page_{i:04d}.txt'

        try:
            # 이미지 다운로드
            img_resp = requests.get(item['imageUrl'], timeout=30)
            if img_resp.status_code != 200:
                print(f'  ⚠️ 이미지 다운로드 실패: {item["imageUrl"][:80]}...')
                continue

            with open(dataset_dir / split / 'images' / img_name, 'wb') as f:
                f.write(img_resp.content)

            # YOLO 라벨 파일 (class_id cx cy w h)
            with open(dataset_dir / split / 'labels' / label_name, 'w') as f:
                f.write(item['labels'])

            success += 1
            if (i + 1) % 50 == 0:
                print(f'  📦 {i + 1}/{len(items)} 다운로드 완료')
        except Exception as e:
            print(f'  ❌ 오류: {e}')

    print(f'\n✅ 다운로드 완료: {success}/{len(items)}개')
    !find dataset -name '*.png' | wc -l
    !find dataset -name '*.txt' | wc -l

In [ ]:
# Cell 3: data.yaml 생성

data_yaml = """path: /content/dataset
train: train/images
val: val/images

nc: 3
names: ['problem', 'graph', 'table']
"""

with open('dataset/data.yaml', 'w') as f:
    f.write(data_yaml)

print('✅ data.yaml 생성 완료')
print(data_yaml)

In [ ]:
# Cell 4: YOLOv8 학습

# nano 모델로 시작 (CPU 추론 속도 우선)
# 데이터 300장+ 시 small 모델로 업그레이드 가능
model = YOLO('yolov8n.pt')  # yolov8s.pt 로 변경 가능

results = model.train(
    data='dataset/data.yaml',
    epochs=100,
    imgsz=1024,          # 시험지 페이지는 큰 이미지 (1024로 디테일 유지)
    batch=8,             # Colab free T4 GPU 기준
    patience=20,         # 조기 종료
    device=0,            # GPU
    project='runs',
    name='math_problem_detector',
    # 문서 감지 특화 augmentation
    flipud=0.0,          # 상하 반전 없음 (텍스트 방향 중요)
    fliplr=0.0,          # 좌우 반전 없음 (읽기 순서 중요)
    mosaic=0.5,          # 모자이크 감소 (문서는 구조화됨)
    degrees=2.0,         # 약간의 회전 (스캔 기울어짐 대응)
    scale=0.3,           # 적당한 스케일
    verbose=True,
)

print('\n✅ 학습 완료!')

In [ ]:
# Cell 5: 평가 + 모델 다운로드

# 검증
best_model_path = 'runs/math_problem_detector/weights/best.pt'
val_model = YOLO(best_model_path)
metrics = val_model.val()

print(f'\n🎯 mAP50: {metrics.box.map50:.4f}')
print(f'🎯 mAP50-95: {metrics.box.map:.4f}')

# 클래스별 mAP
for i, name in enumerate(['problem', 'graph', 'table']):
    try:
        ap = metrics.box.ap50[i]
        print(f'  {name}: AP50={ap:.4f}')
    except:
        pass

print(f'\n💾 모델 경로: {best_model_path}')

# 다운로드
from google.colab import files
files.download(best_model_path)
print('✅ best.pt 다운로드 완료 → yolo-server/models/ 에 배치하세요')

In [ ]:
# Cell 6: 샘플 이미지로 테스트

import glob

test_images = glob.glob('dataset/val/images/*.png')[:3]
if test_images:
    test_model = YOLO(best_model_path)
    for img_path in test_images:
        results = test_model(img_path, conf=0.25)
        print(f'\n🖼 {img_path}: {len(results[0].boxes)} detections')
        for box in results[0].boxes:
            cls = int(box.cls[0])
            conf = box.conf[0].item()
            print(f'  class={["problem","graph","table"][cls]} conf={conf:.3f}')
        results[0].show()  # 시각화
else:
    print('⚠️ 테스트 이미지 없음')

## 재학습 (Fine-tune)

데이터가 더 쌓이면 기존 모델 기반으로 추가 학습:

```python
model = YOLO('best.pt')  # 기존 모델
model.train(
    data='dataset/data.yaml',
    epochs=30,           # 적은 epoch
    imgsz=1024,
    batch=8,
    lr0=0.001,           # 낮은 학습률
    patience=10,
)
```